# IndoML Track 1 — V34.1 Rescue-Only — FIXED NO-LEAK
## Stable posterior pair proposals → missed-event classifier → definite submission
### Natural-only additions; exact best Synthetic branch frozen

This notebook is based directly on the V34 diagnostic:

```text
SELECT stable baseline      F1 = 0.368724
SELECT oracle ADD-only      F1 = 0.589324
TUNE   oracle ADD-only      F1 = 0.560458

SELECT pair proposal "coverage" ≈ 0.624
HOLD   pair proposal "coverage" ≈ 0.609
```

The decisive observation is that **we do not need to rebuild Natural predictions**.

The successful oracle intervention was:

```text
existing baseline events
        KEEP ALL
          +
recover baseline-missed GT events
        ADD ONLY
```

Therefore V34.1 changes the learning target from:

> "Is this candidate any GT event?"

to:

> **"Does this candidate rescue a GT event that the baseline missed?"**

No DROP.  
No boundary replacement.  
No IoU suppression against the baseline.  
Overlapping Natural events are allowed.

The notebook also fixes the V34 candidate-oracle bug. It reports:

1. independent per-GT candidate existence;
2. maximum-cardinality bipartite candidate↔GT coverage;
3. official greedy competition score only for actual prediction sets.

### Validation discipline

```text
TRAIN  buckets 0..3 → classifier fitting only
TUNE   buckets 4..5 → rescue threshold / policy selection
SELECT buckets 6..7 → diagnostic only
HOLD   buckets 8..9 → diagnostic only
```

The submission policy is frozen using **TUNE only**.  
SELECT/HOLD are printed for evidence but never used to choose the hidden-test policy.

### Definite submission behavior

Unlike V34, this notebook does **not** stop on a validation gate.

After fitting and freezing the TUNE-selected rescue policy, the final cells always create:

```text
predictions_Track1_V34_1_RescueOnly_FINAL.jsonl
submission_Track1_V34_1_RescueOnly_FINAL.zip
```

provided the existing required Drive artifacts are available:

- official validation posterior cache;
- test posterior cache;
- exact 1.4979 best submission JSONL;
- existing `p_syn` router JSONL.

For hidden test:
- Synthetic route is copied exactly from the 1.4979 submission.
- Natural route keeps every event from the 1.4979 submission and can only **ADD** high-confidence rescue events.


> **Important fix:** the earlier V34.1 notebook accidentally left
> `matched_missed_count` (a GT-derived diagnostic) inside `FEATURE_COLS`.
> That caused target leakage and invalidated the near-perfect SELECT/HOLD rescue results.
> This notebook explicitly excludes that field and contains a hard leakage assertion.


In [1]:
# ============================================================
# 0 — Setup
# ============================================================

from pathlib import Path
import os, json, pickle, hashlib, glob, zipfile, math, warnings
from collections import Counter, defaultdict, deque

try:
    from google.colab import drive
    if not Path('/content/drive/MyDrive').exists():
        drive.mount('/content/drive')
except Exception:
    pass

import numpy as np
import pandas as pd

from scipy.signal import find_peaks
from sklearn.ensemble import HistGradientBoostingClassifier, ExtraTreesClassifier
from sklearn.metrics import roc_auc_score, average_precision_score

warnings.filterwarnings('ignore')

WORK = Path('/content/drive/MyDrive/Datathon/work')
ROOT = WORK / 'track1_v34_1_rescue_only'
ROOT.mkdir(parents=True, exist_ok=True)

VAL_CACHE = WORK / 'official_val_posteriors_epoch_2.pkl'
TEST_CACHE = WORK / 'test_posteriors_wavlm_beats_epoch2.pkl'

FRAME = 0.010
SEED = 42

# Stable decoder used to construct the released-validation baseline.
BASE_THR = 0.325
BASE_MIN_DUR = 0.060
BASE_SNAP = 0.20

# Proposal generator. Pair proposals were the dominant useful source in V34.
REGION_THRESHOLDS = [0.08,0.12,0.16,0.20,0.25,0.325,0.40,0.50,0.60]
PAIR_REGION_THRESHOLDS = [0.08,0.12,0.20,0.325]
PAIR_WIN_SEC = 0.40
PEAK_MIN_DISTANCE_SEC = 0.04
MAX_BOUNDARY_MINIMA = 18
MIN_EVENT_DUR = 0.050
MAX_EVENT_DUR = 6.0
MAX_CANDIDATES_PER_CLIP = 180
PAIR_MIN_MAX_POST = 0.20
PAIR_MIN_MEAN_POST = 0.05

# Rescue policy. TUNE chooses among these; SELECT/HOLD do not change the choice.
SOURCE_MODES = ['pairs_only','all_nonbaseline']
MAX_ADDS = [1,2,3,4,6]
DUP_SECS = [0.03,0.05,0.08]

# Exact hidden-best identity.
EXPECTED_BEST_SHA256 = 'bee50cc35a306ca46482e092bd842328ae8b39040e004e6a9baee0a2a47a0cfa'
EXPECTED_TEST_CLIPS = 5517
GATE_THR = 0.425

assert VAL_CACHE.is_file(), VAL_CACHE

print('VAL_CACHE :',VAL_CACHE)
print('TEST_CACHE:',TEST_CACHE,'exists:',TEST_CACHE.is_file())
print('ROOT      :',ROOT)


Mounted at /content/drive
VAL_CACHE : /content/drive/MyDrive/Datathon/work/official_val_posteriors_epoch_2.pkl
TEST_CACHE: /content/drive/MyDrive/Datathon/work/test_posteriors_wavlm_beats_epoch2.pkl exists: True
ROOT      : /content/drive/MyDrive/Datathon/work/track1_v34_1_rescue_only


In [2]:
# ============================================================
# 1 — Exact Track-1 scorer
# ============================================================

def matches_official(ev, gt):
    po,pf=ev; ro,rf=gt
    tol=max(.20*(rf-ro),.05)
    return abs(po-ro)<=tol and abs(pf-rf)<=tol

def boundary_error(ev,gt):
    return abs(ev[0]-gt[0])+abs(ev[1]-gt[1])

def match_events(pred, ref):
    cands=[]
    for i,ev in enumerate(pred):
        for j,gt in enumerate(ref):
            if matches_official(ev,gt):
                cands.append((boundary_error(ev,gt),i,j))
    cands.sort()
    up=set(); ur=set(); pairs=[]
    for _,i,j in cands:
        if i in up or j in ur:
            continue
        up.add(i); ur.add(j); pairs.append((i,j))
    return len(pairs),pairs

def event_f1(preds,refs,ids):
    TP=FP=FN=0
    for cid in ids:
        p=preds.get(cid,[])
        r=refs[cid]
        tp,_=match_events(p,r)
        TP+=tp
        FP+=len(p)-tp
        FN+=len(r)-tp
    P=TP/(TP+FP) if TP+FP else 0.
    R=TP/(TP+FN) if TP+FN else 0.
    F=2*P*R/(P+R) if P+R else 0.
    return F,P,R,TP,FP,FN

def _mask(events,n):
    m=np.zeros(n,bool)
    for on,off in events:
        a=max(0,int(round(on/FRAME)))
        b=min(n,int(round(off/FRAME)))
        if b>a:
            m[a:b]=True
    return m

def segment_dice(preds,refs,durs,ids):
    vals=[]
    for cid in ids:
        n=max(1,int(round(durs[cid]/FRAME)))
        P=_mask(preds.get(cid,[]),n)
        G=_mask(refs[cid],n)
        den=P.sum()+G.sum()
        vals.append(1.0 if den==0 else 2*np.logical_and(P,G).sum()/den)
    return float(np.mean(vals))

def score(preds,refs,durs,ids):
    f,p,r,tp,fp,fn=event_f1(preds,refs,ids)
    d=segment_dice(preds,refs,durs,ids)
    return dict(
        f1=float(f),precision=float(p),recall=float(r),
        tp=int(tp),fp=int(fp),fn=int(fn),
        dice=float(d),combined=float(f+d)
    )

def interval_iou(a,b):
    inter=max(0.,min(a[1],b[1])-max(a[0],b[0]))
    union=max(a[1],b[1])-min(a[0],b[0])
    return inter/union if union>0 else 0.

print('official-style scorer ready')


official-style scorer ready


In [3]:
# ============================================================
# 2 — Load released validation cache + leakage-safe splits
# ============================================================

RAW=pickle.load(open(VAL_CACHE,'rb'))
assert isinstance(RAW,list) and len(RAW)>1000

CACHE={}
REFS={}
DURS={}
IS_SYN={}

for r in RAW:
    cid=str(r['clip_id'])
    p=np.asarray(r['prob'],np.float32).reshape(-1)
    on=np.asarray(r['onset_dist'],np.float32).reshape(-1)
    off=np.asarray(r['offset_dist'],np.float32).reshape(-1)
    T=min(len(p),len(on),len(off))
    dur=float(r['duration'])

    gt=[]
    for ev in r.get('gt',[]):
        try:
            a,b=float(ev[0]),float(ev[1])
            a=max(0.,min(a,dur))
            b=max(0.,min(b,dur))
            if b>a:
                gt.append((a,b))
        except Exception:
            pass

    CACHE[cid]=dict(frame=p[:T],onset=on[:T],offset=off[:T],dur=dur)
    REFS[cid]=sorted(gt)
    DURS[cid]=dur
    IS_SYN[cid]=bool(r.get('synthetic',False))

NAT_IDS=sorted(c for c in CACHE if not IS_SYN[c])

def group_key(cid):
    parts=str(cid).split('_')
    return parts[5] if len(parts)>6 else str(cid)

def bucket_group(cid):
    return int(hashlib.md5(group_key(cid).encode()).hexdigest(),16)%10

TRAIN_IDS=[c for c in NAT_IDS if bucket_group(c)<=3]
TUNE_IDS=[c for c in NAT_IDS if bucket_group(c) in {4,5}]
SELECT_IDS=[c for c in NAT_IDS if bucket_group(c) in {6,7}]
HOLD_IDS=[c for c in NAT_IDS if bucket_group(c) in {8,9}]

print('Natural clips/events:',len(NAT_IDS),sum(len(REFS[c]) for c in NAT_IDS))
print('TRAIN / TUNE / SELECT / HOLD:',
      len(TRAIN_IDS),len(TUNE_IDS),len(SELECT_IDS),len(HOLD_IDS))


Natural clips/events: 1806 2944
TRAIN / TUNE / SELECT / HOLD: 673 354 393 386


In [4]:
# ============================================================
# 3 — Stable released-validation baseline
# ============================================================

def regions(mask):
    out=[]; st=None
    for i,v in enumerate(mask):
        if v and st is None:
            st=i
        elif (not v) and st is not None:
            out.append((st,i)); st=None
    if st is not None:
        out.append((st,len(mask)))
    return out

def stable_decode(z,thr=BASE_THR,min_dur=BASE_MIN_DUR,snap=BASE_SNAP):
    p=z['frame']; on=z['onset']; off=z['offset']; dur=z['dur']
    T=len(p)
    W=max(1,int(round(snap/FRAME)))
    out=[]
    for a,b in regions(p>=thr):
        x0=max(0,a-W); x1=min(T,a+W+1)
        sf=x0+int(np.argmin(on[x0:x1])) if x1>x0 else a

        x0=max(0,b-W); x1=min(T,b+W+1)
        ef=x0+int(np.argmin(off[x0:x1])) if x1>x0 else b

        s=max(0.,sf*FRAME)
        e=min(dur,ef*FRAME)
        if e-s>=min_dur:
            out.append((s,e))
    return sorted(out)

BASE_PRED={cid:stable_decode(CACHE[cid]) for cid in NAT_IDS}

rows=[]
for name,ids in [('TRAIN',TRAIN_IDS),('TUNE',TUNE_IDS),('SELECT',SELECT_IDS),('HOLD',HOLD_IDS)]:
    rows.append({'split':name,**score(BASE_PRED,REFS,DURS,ids)})
BASE_TABLE=pd.DataFrame(rows)
display(BASE_TABLE)


,split,f1,precision,recall,tp,fp,fn,dice,combined
0,TRAIN,0.355663,0.384375,0.330942,369,591,746,0.709364,1.065027
1,TUNE,0.315488,0.343750,0.291519,165,315,401,0.681743,0.997231
2,SELECT,0.368724,0.388889,0.350548,224,352,415,0.689910,1.058634
3,HOLD,0.349650,0.384615,0.320513,200,320,424,0.686311,1.035961


In [5]:
# ============================================================
# 4 — High-recall pair candidate generator
# ============================================================

def local_minima(x,max_n=MAX_BOUNDARY_MINIMA,min_dist_sec=PEAK_MIN_DISTANCE_SEC):
    x=np.asarray(x,float)
    if not len(x):
        return []
    dist=max(1,int(round(min_dist_sec/FRAME)))
    pk,_=find_peaks(-x,distance=dist)
    if len(pk)==0:
        pk=np.arange(len(x))
    pk=sorted(pk,key=lambda i:float(x[i]))
    extra=[int(np.argmin(x)),0,len(x)-1]
    out=[]
    for i in pk+extra:
        i=int(i)
        if i not in out:
            out.append(i)
        if len(out)>=max_n:
            break
    return sorted(out)

def nearest_minima(x,center,win_sec,k=4):
    T=len(x)
    W=max(1,int(round(win_sec/FRAME)))
    a=max(0,center-W)
    b=min(T,center+W+1)
    if b<=a:
        return [max(0,min(T-1,center))]
    idx=np.arange(a,b)
    vals=np.asarray(x[a:b])
    order=np.argsort(vals)[:min(k,len(vals))]
    return sorted(set(int(idx[o]) for o in order))

def interval_stats(cache,cid,s,e):
    z=cache[cid]
    p=z['frame']; T=len(p)
    a=max(0,min(T-1,int(round(s/FRAME))))
    b=max(a+1,min(T,int(round(e/FRAME))))
    seg=p[a:b]
    return (
        float(np.max(seg)) if len(seg) else 0.,
        float(np.mean(seg)) if len(seg) else 0.,
    )

def add_candidate(store,cache,cid,s,e,source,seed_score):
    dur=cache[cid]['dur']
    s=max(0.,min(float(s),dur))
    e=max(0.,min(float(e),dur))
    if e-s<MIN_EVENT_DUR or e-s>MAX_EVENT_DUR:
        return
    key=(round(s,3),round(e,3))
    if key not in store:
        store[key]=dict(
            onset=s,offset=e,source=source,seed_score=float(seed_score)
        )
    else:
        old=store[key]
        src=set(old['source'].split('|'))|{source}
        old['source']='|'.join(sorted(src))
        old['seed_score']=max(old['seed_score'],float(seed_score))

def near_same(a,b,sec=.02):
    return abs(a[0]-b[0])<=sec and abs(a[1]-b[1])<=sec

def dedupe_candidates(cands,max_n=MAX_CANDIDATES_PER_CLIP):
    # Pair proposals are preferred when cap pressure exists.
    def priority(z):
        src=z['source'].split('|')
        pair=int(any(s.startswith('pair_') or s=='global_pair' for s in src))
        return (-pair,-z.get('seed_score',0.),-len(src),z['onset'])
    kept=[]
    for z in sorted(cands,key=priority):
        ev=(z['onset'],z['offset'])
        if any(near_same(ev,(x['onset'],x['offset']),.02) for x in kept):
            continue
        kept.append(z)
        if len(kept)>=max_n:
            break
    return sorted(kept,key=lambda z:(z['onset'],z['offset']))

def generate_candidates(cache,cid,baseline_events):
    z=cache[cid]
    p,on,off,dur=z['frame'],z['onset'],z['offset'],z['dur']
    T=len(p)
    store={}

    # Baseline is recorded only so source/features know it exists.
    for s,e in baseline_events:
        mx,me=interval_stats(cache,cid,s,e)
        add_candidate(store,cache,cid,s,e,'baseline',mx+me)

    # Multi-threshold connected regions + local boundary combinations.
    for thr in REGION_THRESHOLDS:
        for a,b in regions(p>=thr):
            if b<=a:
                continue

            W=max(1,int(round(BASE_SNAP/FRAME)))
            x0=max(0,a-W); x1=min(T,a+W+1)
            sf=x0+int(np.argmin(on[x0:x1])) if x1>x0 else a

            x0=max(0,b-W); x1=min(T,b+W+1)
            ef=x0+int(np.argmin(off[x0:x1])) if x1>x0 else b

            s=sf*FRAME; e=min(dur,ef*FRAME)
            mx,me=interval_stats(cache,cid,s,e)
            add_candidate(store,cache,cid,s,e,f'region_{thr:.3f}',mx+me)

            if thr in PAIR_REGION_THRESHOLDS:
                oi=nearest_minima(on,a,PAIR_WIN_SEC,k=4)
                fi=nearest_minima(off,max(a+1,b-1),PAIR_WIN_SEC,k=4)
                for x in oi:
                    for y in fi:
                        if y<=x:
                            continue
                        s=x*FRAME; e=min(dur,y*FRAME)
                        mx,me=interval_stats(cache,cid,s,e)
                        if mx>=PAIR_MIN_MAX_POST and me>=PAIR_MIN_MEAN_POST:
                            add_candidate(store,cache,cid,s,e,f'pair_{thr:.3f}',mx+me)

    # Global onset-offset combinations recover split / overlap cases.
    ons=local_minima(on)
    offs=local_minima(off)
    for oi in ons:
        for fi in offs:
            if fi<=oi:
                continue
            s=oi*FRAME
            e=min(dur,fi*FRAME)
            d=e-s
            if d<MIN_EVENT_DUR or d>MAX_EVENT_DUR:
                continue
            mx,me=interval_stats(cache,cid,s,e)
            if mx>=PAIR_MIN_MAX_POST and me>=PAIR_MIN_MEAN_POST:
                add_candidate(store,cache,cid,s,e,'global_pair',mx+me)

    return dedupe_candidates(list(store.values()))

CAND={}
for n,cid in enumerate(NAT_IDS,1):
    CAND[cid]=generate_candidates(CACHE,cid,BASE_PRED[cid])
    if n%500==0:
        print('generated',n,'/',len(NAT_IDS))

cnt=np.asarray([len(CAND[c]) for c in NAT_IDS])
print('candidate mean/median/q90/max:',
      cnt.mean(),np.median(cnt),np.quantile(cnt,.90),cnt.max())


generated 500 / 1806
generated 1000 / 1806
generated 1500 / 1806
candidate mean/median/q90/max: 101.8859357696567 96.0 180.0 180


## Corrected oracle

The original V34 `coverage` used the official greedy scorer on a huge candidate set.
That is not a pure proposal-coverage metric because duplicate candidates can compete for GT events.

This notebook therefore reports two true proposal diagnostics:

- **Existence coverage:** each GT independently asks whether at least one candidate can match it.
- **Maximum bipartite coverage:** maximum number of GT events that can be covered one-to-one by distinct candidates.

The final leaderboard-style F1 still uses the official greedy matcher.


In [6]:
# ============================================================
# 5 — Correct candidate coverage: existence + max bipartite matching
# ============================================================

def hopcroft_karp(adj,n_left,n_right):
    # left=candidates, right=GT
    NIL=-1
    pair_u=[NIL]*n_left
    pair_v=[NIL]*n_right
    dist=[0]*n_left

    def bfs():
        q=deque()
        found=False
        for u in range(n_left):
            if pair_u[u]==NIL:
                dist[u]=0
                q.append(u)
            else:
                dist[u]=-1
        while q:
            u=q.popleft()
            for v in adj[u]:
                pu=pair_v[v]
                if pu==NIL:
                    found=True
                elif dist[pu]<0:
                    dist[pu]=dist[u]+1
                    q.append(pu)
        return found

    def dfs(u):
        for v in adj[u]:
            pu=pair_v[v]
            if pu==NIL or (dist[pu]==dist[u]+1 and dfs(pu)):
                pair_u[u]=v
                pair_v[v]=u
                return True
        dist[u]=-1
        return False

    matching=0
    while bfs():
        for u in range(n_left):
            if pair_u[u]==NIL and dfs(u):
                matching+=1
    return matching,pair_u,pair_v

def proposal_events(cid,mode='union'):
    out=[]
    for z in CAND[cid]:
        src=z['source'].split('|')
        pair=any(s.startswith('pair_') or s=='global_pair' for s in src)
        baseline='baseline' in src
        region=any(s.startswith('region_') for s in src)
        if mode=='pairs' and not pair:
            continue
        if mode=='baseline' and not baseline:
            continue
        if mode=='regions' and not (baseline or region):
            continue
        out.append((z['onset'],z['offset']))
    return out

def corrected_oracle(ids,mode='union'):
    gt_total=exist=matching_total=0
    overlap_total=overlap_exist=0

    for cid in ids:
        cand=proposal_events(cid,mode)
        gt=REFS[cid]
        gt_total+=len(gt)

        overlap_flags=[]
        for j,g in enumerate(gt):
            overlap_flags.append(any(
                j!=k and min(g[1],h[1])-max(g[0],h[0])>0
                for k,h in enumerate(gt)
            ))

        for j,g in enumerate(gt):
            hit=any(matches_official(c,g) for c in cand)
            exist+=int(hit)
            if overlap_flags[j]:
                overlap_total+=1
                overlap_exist+=int(hit)

        adj=[]
        for c in cand:
            adj.append([j for j,g in enumerate(gt) if matches_official(c,g)])
        m,_,_=hopcroft_karp(adj,len(cand),len(gt))
        matching_total+=m

    existence=exist/gt_total if gt_total else 0.
    maxmatch=matching_total/gt_total if gt_total else 0.
    over=overlap_exist/overlap_total if overlap_total else np.nan
    oracle_f1=2*maxmatch/(1+maxmatch) if maxmatch else 0.

    return dict(
        gt_events=gt_total,
        existence_coverage=existence,
        max_bipartite_coverage=maxmatch,
        oracle_selected_f1_ceiling=oracle_f1,
        overlap_existence_coverage=over,
    )

oracle_rows=[]
for split,ids in [('TRAIN',TRAIN_IDS),('TUNE',TUNE_IDS),('SELECT',SELECT_IDS),('HOLD',HOLD_IDS)]:
    for mode in ['baseline','regions','pairs','union']:
        oracle_rows.append({'split':split,'source':mode,**corrected_oracle(ids,mode)})

CORRECTED_ORACLE=pd.DataFrame(oracle_rows)
display(CORRECTED_ORACLE)
CORRECTED_ORACLE.to_csv(ROOT/'corrected_candidate_oracle.csv',index=False)


,split,source,gt_events,existence_coverage,max_bipartite_coverage,oracle_selected_f1_ceiling,overlap_existence_coverage
0,TRAIN,baseline,1115,0.094170,0.092377,0.169130,0.068027
1,TRAIN,regions,1115,0.210762,0.208072,0.344469,0.180272
2,TRAIN,pairs,1115,0.617040,0.617040,0.763172,0.656463
3,TRAIN,union,1115,0.629596,0.628700,0.772026,0.670068
4,TUNE,baseline,566,0.068905,0.068905,0.128926,0.043165
5,TUNE,regions,566,0.224382,0.224382,0.366522,0.172662
6,TUNE,pairs,566,0.607774,0.607774,0.756044,0.647482
7,TUNE,union,566,0.620141,0.620141,0.765540,0.676259
8,SELECT,baseline,639,0.087637,0.086072,0.158501,0.084848
9,SELECT,regions,639,0.205008,0.203443,0.338101,0.193939


In [7]:
# ============================================================
# 6 — ADD-only oracle on ALL validation partitions, including HOLD
# ============================================================

def oracle_add_only(ids):
    pred={}
    for cid in ids:
        base=list(BASE_PRED[cid])
        gt=REFS[cid]

        # Officially match baseline first: only unmatched GT are rescue targets.
        _,pairs=match_events(base,gt)
        missed=[j for j in range(len(gt)) if j not in {j for _,j in pairs}]

        out=list(base)
        used=set()

        # For each missed GT choose the closest valid non-baseline candidate.
        for j in missed:
            g=gt[j]
            elig=[]
            for k,z in enumerate(CAND[cid]):
                src=z['source'].split('|')
                if 'baseline' in src:
                    continue
                ev=(z['onset'],z['offset'])
                if matches_official(ev,g):
                    elig.append((boundary_error(ev,g),k,ev))
            if elig:
                _,k,ev=min(elig)
                if k not in used:
                    out.append(ev)
                    used.add(k)

        pred[cid]=out
    return pred

oracle_add_rows=[]
for split,ids in [('TRAIN',TRAIN_IDS),('TUNE',TUNE_IDS),('SELECT',SELECT_IDS),('HOLD',HOLD_IDS)]:
    b=score(BASE_PRED,REFS,DURS,ids)
    a=score(oracle_add_only(ids),REFS,DURS,ids)
    oracle_add_rows.append({'split':split,'system':'baseline',**b})
    oracle_add_rows.append({'split':split,'system':'oracle_add_only',**a})

ADD_ORACLE=pd.DataFrame(oracle_add_rows)
display(ADD_ORACLE)
ADD_ORACLE.to_csv(ROOT/'add_only_oracle.csv',index=False)


,split,system,f1,precision,recall,tp,fp,fn,dice,combined
0,TRAIN,baseline,0.355663,0.384375,0.330942,369,591,746,0.709364,1.065027
1,TRAIN,oracle_add_only,0.585406,0.544333,0.633184,706,591,409,0.769706,1.355112
2,TUNE,baseline,0.315488,0.343750,0.291519,165,315,401,0.681743,0.997231
3,TUNE,oracle_add_only,0.569805,0.527027,0.620141,351,315,215,0.747268,1.317073
4,SELECT,baseline,0.368724,0.388889,0.350548,224,352,415,0.689910,1.058634
5,SELECT,oracle_add_only,0.589324,0.540470,0.647887,414,352,225,0.759828,1.349152
6,HOLD,baseline,0.349650,0.384615,0.320513,200,320,424,0.686311,1.035961
7,HOLD,oracle_add_only,0.585768,0.549930,0.626603,391,320,233,0.757603,1.343371


## Rescue labels

For each clip:

1. match baseline predictions to GT using the official scorer;
2. identify GT events missed by the baseline;
3. consider only non-baseline proposals;
4. label a proposal positive if it matches at least one **missed** GT.

Positive sample weights are divided across duplicate candidates matching the same missed GT,
so one easy GT with many near-equivalent proposals cannot dominate training.


In [8]:
# ============================================================
# 7 — Rescue-only features and labels
# ============================================================

SOURCE_NAMES = (
    [f'region_{t:.3f}' for t in REGION_THRESHOLDS] +
    [f'pair_{t:.3f}' for t in PAIR_REGION_THRESHOLDS] +
    ['global_pair']
)

def q(x,q):
    return float(np.quantile(x,q)) if len(x) else 0.

def baseline_relation(ev,base):
    if not base:
        return dict(
            base_max_iou=0.,base_overlap_count=0,
            near_base_on=99.,near_base_off=99.,
            near_base_both=99.,inside_base=0,contains_base=0,
            duration_ratio_near=0.
        )

    ious=[interval_iou(ev,b) for b in base]
    near_on=[abs(ev[0]-b[0]) for b in base]
    near_off=[abs(ev[1]-b[1]) for b in base]
    both=[near_on[i]+near_off[i] for i in range(len(base))]
    k=int(np.argmin(both))
    bd=max(base[k][1]-base[k][0],1e-4)

    return dict(
        base_max_iou=float(max(ious)),
        base_overlap_count=int(sum(x>0 for x in ious)),
        near_base_on=float(min(near_on)),
        near_base_off=float(min(near_off)),
        near_base_both=float(min(both)),
        inside_base=int(any(ev[0]>=b[0] and ev[1]<=b[1] for b in base)),
        contains_base=int(any(ev[0]<=b[0] and ev[1]>=b[1] for b in base)),
        duration_ratio_near=float((ev[1]-ev[0])/bd),
    )

def feature_row(cache,cid,z,base,cand_count):
    rec=cache[cid]
    p,on,off,dur=rec['frame'],rec['onset'],rec['offset'],rec['dur']
    T=len(p)

    s=float(z['onset']); e=float(z['offset'])
    ev=(s,e)
    a=max(0,min(T-1,int(round(s/FRAME))))
    b=max(a+1,min(T,int(round(e/FRAME))))
    off_idx=min(T-1,b)

    seg=p[a:b]
    pre=p[max(0,a-25):a]
    post=p[b:min(T,b+25)]
    bg=np.r_[pre,post] if len(pre)+len(post) else p

    W=max(1,int(round(.15/FRAME)))
    onwin=on[max(0,a-W):min(T,a+W+1)]
    offwin=off[max(0,off_idx-W):min(T,off_idx+W+1)]

    # Posterior shape / number of strong islands inside candidate.
    strong=(seg>=BASE_THR) if len(seg) else np.zeros(0,bool)
    island_count=len(regions(strong)) if len(strong) else 0

    src=set(z['source'].split('|'))
    rel=baseline_relation(ev,base)

    row=dict(
        cid=cid,
        onset=s,offset=e,
        duration=e-s,
        duration_frac=(e-s)/max(dur,1e-6),
        onset_frac=s/max(dur,1e-6),
        offset_frac=e/max(dur,1e-6),

        p_max=float(np.max(seg)) if len(seg) else 0.,
        p_mean=float(np.mean(seg)) if len(seg) else 0.,
        p_min=float(np.min(seg)) if len(seg) else 0.,
        p_q10=q(seg,.10),p_q25=q(seg,.25),p_q50=q(seg,.50),
        p_q75=q(seg,.75),p_q90=q(seg,.90),

        bg_mean=float(np.mean(bg)) if len(bg) else 0.,
        bg_q90=q(bg,.90),bg_q95=q(bg,.95),
        contrast_mean=(float(np.mean(seg))-float(np.mean(bg))) if len(seg) and len(bg) else 0.,
        contrast_q90=(q(seg,.90)-q(bg,.95)) if len(seg) and len(bg) else 0.,
        occupancy_above_0325=float(np.mean(seg>=.325)) if len(seg) else 0.,
        occupancy_above_050=float(np.mean(seg>=.50)) if len(seg) else 0.,
        strong_islands=int(island_count),

        onset_value=float(on[a]),
        offset_value=float(off[off_idx]),
        onset_local_min=float(np.min(onwin)) if len(onwin) else float(on[a]),
        offset_local_min=float(np.min(offwin)) if len(offwin) else float(off[off_idx]),
        onset_global_pct=float(np.mean(on<=on[a])),
        offset_global_pct=float(np.mean(off<=off[off_idx])),

        seed_score=float(z.get('seed_score',0.)),
        source_count=len(src),
        is_pair=int(any(x.startswith('pair_') for x in src)),
        is_global_pair=int('global_pair' in src),
        is_region=int(any(x.startswith('region_') for x in src)),
        candidate_count=int(cand_count),
        baseline_count=int(len(base)),
        **rel,
    )

    for sn in SOURCE_NAMES:
        row['src_'+sn]=int(sn in src)
    return row

def missed_gt_indices(cid,base):
    gt=REFS[cid]
    _,pairs=match_events(base,gt)
    hit={j for _,j in pairs}
    return [j for j in range(len(gt)) if j not in hit]

SPLIT_OF={}
for c in TRAIN_IDS: SPLIT_OF[c]='TRAIN'
for c in TUNE_IDS: SPLIT_OF[c]='TUNE'
for c in SELECT_IDS: SPLIT_OF[c]='SELECT'
for c in HOLD_IDS: SPLIT_OF[c]='HOLD'

rows=[]

for cid in NAT_IDS:
    base=BASE_PRED[cid]
    missed=missed_gt_indices(cid,base)

    # Build only non-baseline candidate rows and match maps.
    proposed=[]
    for j,z in enumerate(CAND[cid]):
        src=set(z['source'].split('|'))
        if 'baseline' in src:
            continue
        ev=(z['onset'],z['offset'])

        # Do not train on near-exact baseline duplicates: they cannot rescue a new event.
        if any(near_same(ev,b,.03) for b in base):
            continue

        matched=[g for g in missed if matches_official(ev,REFS[cid][g])]
        proposed.append((j,z,matched))

    # Number of proposal candidates that can explain each missed GT.
    counts=Counter(g for _,_,ms in proposed for g in ms)

    for j,z,matched in proposed:
        r=feature_row(CACHE,cid,z,base,len(proposed))
        r['cand_idx']=j
        r['split']=SPLIT_OF[cid]
        r['label']=int(bool(matched))
        r['matched_missed_count']=len(matched)

        if matched:
            # prevent one GT with 30 near-duplicates from dominating.
            r['base_sample_weight']=max(1.0/counts[g] for g in matched)
        else:
            r['base_sample_weight']=1.0

        rows.append(r)

FEAT=pd.DataFrame(rows)

NON_FEATURE={
    'cid','cand_idx','split','label','onset','offset',
    'base_sample_weight',
    # CRITICAL: this is computed from GT and is ONLY for diagnostics/weighting.
    # It must never be visible to the classifier.
    'matched_missed_count'
}
FEATURE_COLS=[c for c in FEAT.columns if c not in NON_FEATURE]

# Hard leakage guard: fail if any GT-derived target metadata leaks into model features.
FORBIDDEN_FEATURE_TOKENS = ('label','matched','missed','ground_truth','groundtruth','gt_','target')
leaky = [c for c in FEATURE_COLS if any(tok in c.lower() for tok in FORBIDDEN_FEATURE_TOKENS)]
assert not leaky, f'TARGET LEAKAGE DETECTED IN FEATURES: {leaky}'
assert 'matched_missed_count' not in FEATURE_COLS
assert 'base_sample_weight' not in FEATURE_COLS

print('rows:',len(FEAT),'features:',len(FEATURE_COLS))
print('Leakage guard: PASS')
display(FEAT.groupby('split').label.agg(['count','sum','mean']))

FEAT.to_pickle(ROOT/'rescue_candidate_features.pkl')


rows: 175623 features: 55
Leakage guard: PASS


,count,sum,mean
split,,,
HOLD,36236,1056,0.029142
SELECT,38094,1054,0.027668
TRAIN,66840,1871,0.027992
TUNE,34453,1110,0.032218


In [9]:
# ============================================================
# 8 — Fit two complementary rescue classifiers on TRAIN only
# ============================================================

train=FEAT[FEAT.split=='TRAIN'].copy()
X=train[FEATURE_COLS].replace([np.inf,-np.inf],np.nan).fillna(0).to_numpy(np.float32)
y=train.label.to_numpy(np.int64)

pos=max(1,int(y.sum()))
neg=max(1,int((1-y).sum()))
class_boost=min(25.0,neg/pos)

# duplicate-aware positive weighting + overall class balancing
w=train.base_sample_weight.to_numpy(np.float64)
w=np.where(y==1,w*class_boost,w)

hgb=HistGradientBoostingClassifier(
    learning_rate=.045,
    max_iter=260,
    max_leaf_nodes=31,
    min_samples_leaf=24,
    l2_regularization=.8,
    random_state=SEED,
)
hgb.fit(X,y,sample_weight=w)

et=ExtraTreesClassifier(
    n_estimators=500,
    max_depth=18,
    min_samples_leaf=3,
    max_features=.75,
    class_weight='balanced',
    n_jobs=-1,
    random_state=SEED,
)
et.fit(X,y,sample_weight=train.base_sample_weight.to_numpy(np.float64))

def predict_rescue_proba(df):
    X=df[FEATURE_COLS].replace([np.inf,-np.inf],np.nan).fillna(0).to_numpy(np.float32)
    p1=hgb.predict_proba(X)[:,1]
    p2=et.predict_proba(X)[:,1]
    # Fixed ensemble; not selected on SELECT/HOLD.
    return .60*p1+.40*p2

for sp in ['TRAIN','TUNE','SELECT','HOLD']:
    g=FEAT[FEAT.split==sp]
    yy=g.label.to_numpy()
    pp=predict_rescue_proba(g)
    auc=roc_auc_score(yy,pp) if len(np.unique(yy))>1 else np.nan
    ap=average_precision_score(yy,pp) if len(np.unique(yy))>1 else np.nan
    print(
        sp,
        'AUC',round(float(auc),4),
        'PR-AUC',round(float(ap),4),
        'prevalence',round(float(yy.mean()),4),
        'n',len(g)
    )

pickle.dump(
    dict(hgb=hgb,et=et,features=FEATURE_COLS),
    open(ROOT/'v34_1_rescue_models.pkl','wb')
)


TRAIN AUC 0.9995 PR-AUC 0.9806 prevalence 0.028 n 66840
TUNE AUC 0.8255 PR-AUC 0.1579 prevalence 0.0322 n 34453
SELECT AUC 0.7692 PR-AUC 0.1048 prevalence 0.0277 n 38094
HOLD AUC 0.7954 PR-AUC 0.1193 prevalence 0.0291 n 36236


In [10]:
# ============================================================
# 9 — Build per-clip scored rescue proposals
# ============================================================

SCORED={}

for cid in NAT_IDS:
    g=FEAT[FEAT.cid==cid].copy()
    if not len(g):
        SCORED[cid]=[]
        continue

    pp=predict_rescue_proba(g)
    arr=[]
    for (_,r),p in zip(g.iterrows(),pp):
        z=CAND[cid][int(r.cand_idx)]
        ev=(float(z['onset']),float(z['offset']))
        src=set(z['source'].split('|'))
        arr.append(dict(
            event=ev,
            prob=float(p),
            source=z['source'],
            is_pair=any(s.startswith('pair_') or s=='global_pair' for s in src),
            is_global_pair='global_pair' in src,
        ))
    SCORED[cid]=arr

print('scored clips:',len(SCORED))


scored clips: 1806


In [11]:
# ============================================================
# 10 — ADD-only decoder
# No DROP, no REPLACE, no ordinary IoU NMS against baseline.
# ============================================================

def source_allowed(z,mode):
    if mode=='pairs_only':
        return bool(z['is_pair'])
    if mode=='all_nonbaseline':
        return True
    raise ValueError(mode)

def duplicate_of_any(ev,events,dup_sec):
    return any(near_same(ev,x,dup_sec) for x in events)

def rescue_decode(base,scored,cfg):
    out=list(base)
    additions=[]

    for z in sorted(scored,key=lambda x:x['prob'],reverse=True):
        if z['prob']<cfg['threshold']:
            continue
        if not source_allowed(z,cfg['source_mode']):
            continue

        ev=z['event']

        # only near-exact duplicate suppression; genuine overlaps survive
        if duplicate_of_any(ev,out,cfg['dup_sec']):
            continue
        if duplicate_of_any(ev,[x['event'] for x in additions],cfg['dup_sec']):
            continue

        additions.append(z)

        if len(additions)>=cfg['max_add']:
            break

    return sorted(out+[x['event'] for x in additions])

def decode_ids(ids,cfg):
    return {
        cid:rescue_decode(BASE_PRED[cid],SCORED[cid],cfg)
        for cid in ids
    }

print('rescue-only decoder ready')


rescue-only decoder ready


In [12]:
# ============================================================
# 11 — Freeze policy using TUNE only
# ============================================================

# Probability thresholds are derived from the TUNE score distribution so that
# calibration scale does not have to be guessed.
tune_probs=np.asarray([
    z['prob'] for cid in TUNE_IDS for z in SCORED[cid]
],np.float64)

fixed=np.asarray([.10,.15,.20,.25,.30,.35,.40,.45,.50,.55,.60,.65,.70,.75,.80,.85,.90,.93,.95,.97])
quant=np.quantile(tune_probs,np.linspace(.50,.995,36)) if len(tune_probs) else fixed
THRESHOLDS=np.unique(np.clip(np.r_[fixed,quant],0.01,.995))

policy_rows=[]

for mode in SOURCE_MODES:
    for mx in MAX_ADDS:
        for dup in DUP_SECS:
            for thr in THRESHOLDS:
                cfg=dict(
                    source_mode=mode,
                    max_add=int(mx),
                    dup_sec=float(dup),
                    threshold=float(thr),
                )
                pred=decode_ids(TUNE_IDS,cfg)
                s=score(pred,REFS,DURS,TUNE_IDS)
                policy_rows.append({**cfg,**s})

POLICIES=pd.DataFrame(policy_rows).sort_values(
    ['f1','combined','precision'],ascending=False
).reset_index(drop=True)

best=POLICIES.iloc[0]
FROZEN_POLICY=dict(
    source_mode=str(best.source_mode),
    max_add=int(best.max_add),
    dup_sec=float(best.dup_sec),
    threshold=float(best.threshold),
)

print('FROZEN TUNE-ONLY POLICY:')
print(FROZEN_POLICY)
display(POLICIES.head(25))

POLICIES.to_csv(ROOT/'tune_policy_grid.csv',index=False)
json.dump(FROZEN_POLICY,open(ROOT/'frozen_policy.json','w'),indent=2)


FROZEN TUNE-ONLY POLICY:
{'source_mode': 'pairs_only', 'max_add': 3, 'dup_sec': 0.08, 'threshold': 0.6}


,source_mode,max_add,dup_sec,threshold,f1,precision,recall,tp,fp,fn,dice,combined
0,pairs_only,3,0.08,0.60000,0.344335,0.347748,0.340989,193,362,373,0.681721,1.026056
1,all_nonbaseline,3,0.08,0.60000,0.344335,0.347748,0.340989,193,362,373,0.681721,1.026056
2,pairs_only,4,0.08,0.60000,0.343722,0.346499,0.340989,193,364,373,0.681721,1.025443
3,all_nonbaseline,4,0.08,0.60000,0.343722,0.346499,0.340989,193,364,373,0.681721,1.025443
4,pairs_only,2,0.08,0.60000,0.343470,0.347826,0.339223,192,360,374,0.681721,1.025191
5,all_nonbaseline,2,0.08,0.60000,0.343470,0.347826,0.339223,192,360,374,0.681721,1.025191
6,pairs_only,6,0.08,0.60000,0.343416,0.345878,0.340989,193,365,373,0.681721,1.025137
7,pairs_only,3,0.05,0.60000,0.343111,0.345259,0.340989,193,366,373,0.681721,1.024832
8,all_nonbaseline,3,0.05,0.60000,0.343111,0.345259,0.340989,193,366,373,0.681721,1.024832
9,all_nonbaseline,6,0.08,0.60000,0.343111,0.345259,0.340989,193,366,373,0.681721,1.024832


In [13]:
# ============================================================
# 12 — Diagnostic SELECT + HOLD with frozen TUNE policy
# These results DO NOT alter FROZEN_POLICY.
# ============================================================

EVAL_ROWS=[]

for split,ids in [
    ('TUNE',TUNE_IDS),
    ('SELECT',SELECT_IDS),
    ('HOLD',HOLD_IDS),
]:
    b=score(BASE_PRED,REFS,DURS,ids)
    pred=decode_ids(ids,FROZEN_POLICY)
    v=score(pred,REFS,DURS,ids)

    EVAL_ROWS.append({'split':split,'system':'baseline',**b})
    EVAL_ROWS.append({'split':split,'system':'V34.1_rescue_only',**v})

EVAL=pd.DataFrame(EVAL_ROWS)
display(EVAL)
EVAL.to_csv(ROOT/'validation_frozen_policy.csv',index=False)

for split in ['TUNE','SELECT','HOLD']:
    b=EVAL[(EVAL.split==split)&(EVAL.system=='baseline')].iloc[0]
    v=EVAL[(EVAL.split==split)&(EVAL.system=='V34.1_rescue_only')].iloc[0]
    print(
        split,
        'F1 gain',round(float(v.f1-b.f1),4),
        '| TP',int(b.tp),'->',int(v.tp),
        '| FP',int(b.fp),'->',int(v.fp),
        '| FN',int(b.fn),'->',int(v.fn),
    )


,split,system,f1,precision,recall,tp,fp,fn,dice,combined
0,TUNE,baseline,0.315488,0.343750,0.291519,165,315,401,0.681743,0.997231
1,TUNE,V34.1_rescue_only,0.344335,0.347748,0.340989,193,362,373,0.681721,1.026056
2,SELECT,baseline,0.368724,0.388889,0.350548,224,352,415,0.689910,1.058634
3,SELECT,V34.1_rescue_only,0.385938,0.385335,0.386541,247,394,392,0.689964,1.075901
4,HOLD,baseline,0.349650,0.384615,0.320513,200,320,424,0.686311,1.035961
5,HOLD,V34.1_rescue_only,0.365000,0.380208,0.350962,219,357,405,0.688119,1.053119


TUNE F1 gain 0.0288 | TP 165 -> 193 | FP 315 -> 362 | FN 401 -> 373
SELECT F1 gain 0.0172 | TP 224 -> 247 | FP 352 -> 394 | FN 415 -> 392
HOLD F1 gain 0.0153 | TP 200 -> 219 | FP 320 -> 357 | FN 424 -> 405


In [14]:
# ============================================================
# 13 — Topology diagnostics under frozen policy
# ============================================================

def gt_overlap_flags(cid):
    gt=REFS[cid]
    out=[]
    for i,g in enumerate(gt):
        out.append(any(
            i!=j and min(g[1],h[1])-max(g[0],h[0])>0
            for j,h in enumerate(gt)
        ))
    return out

def subset(ids,mode):
    out=[]
    for cid in ids:
        gt=REFS[cid]
        flags=gt_overlap_flags(cid)
        if mode=='overlap' and any(flags):
            out.append(cid)
        elif mode=='nonoverlap' and not any(flags):
            out.append(cid)
        elif mode=='1event' and len(gt)==1:
            out.append(cid)
        elif mode=='2event' and len(gt)==2:
            out.append(cid)
        elif mode=='3plus' and len(gt)>=3:
            out.append(cid)
    return out

rows=[]

for split,ids in [('SELECT',SELECT_IDS),('HOLD',HOLD_IDS)]:
    pred=decode_ids(ids,FROZEN_POLICY)
    for mode in ['overlap','nonoverlap','1event','2event','3plus']:
        ss=subset(ids,mode)
        if not ss:
            continue
        b=score(BASE_PRED,REFS,DURS,ss)
        v=score(pred,REFS,DURS,ss)
        rows.append(dict(
            split=split,subset=mode,n_clips=len(ss),
            base_f1=b['f1'],v34_1_f1=v['f1'],gain=v['f1']-b['f1'],
            base_precision=b['precision'],v34_1_precision=v['precision'],
            base_recall=b['recall'],v34_1_recall=v['recall'],
        ))

TOPO=pd.DataFrame(rows)
display(TOPO)
TOPO.to_csv(ROOT/'topology_diagnostics.csv',index=False)


,split,subset,n_clips,base_f1,v34_1_f1,gain,base_precision,v34_1_precision,base_recall,v34_1_recall
0,SELECT,overlap,70,0.288660,0.345048,0.056388,0.396226,0.421875,0.227027,0.291892
1,SELECT,nonoverlap,323,0.393939,0.399173,0.005233,0.387234,0.376218,0.400881,0.425110
2,SELECT,1event,228,0.390244,0.395303,0.005059,0.363636,0.356890,0.421053,0.442982
3,SELECT,2event,107,0.342857,0.364532,0.021675,0.385965,0.385417,0.308411,0.345794
4,SELECT,3plus,56,0.369048,0.400000,0.030952,0.446043,0.441718,0.314721,0.365482
5,HOLD,overlap,60,0.291667,0.329502,0.037835,0.437500,0.425743,0.218750,0.268750
6,HOLD,nonoverlap,326,0.365044,0.374867,0.009823,0.375000,0.370526,0.355603,0.379310
7,HOLD,1event,221,0.358650,0.350305,-0.008344,0.335968,0.318519,0.384615,0.389140
8,HOLD,2event,113,0.354756,0.382353,0.027597,0.423313,0.428571,0.305310,0.345133
9,HOLD,3plus,52,0.327402,0.365449,0.038046,0.442308,0.443548,0.259887,0.310734


# Hidden test: definite packaging

From this point onward there is no validation gate.

The frozen policy above is applied to the hidden Natural route.

Crucially, the hidden baseline is **not** the local stable decoder.
It is the exact leaderboard-best `1.4979` prediction JSONL.

Therefore hidden inference is:

```text
exact 1.4979 Natural events
        KEEP ALL
           +
stable-posterior rescue proposals
        classifier
           +
TUNE-frozen ADD-only policy
```

Synthetic-routed rows are copied unchanged from the exact 1.4979 file.


In [15]:
# ============================================================
# 14 — Locate exact best submission + router + normalize test cache
# ============================================================

def sha256_file(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for b in iter(lambda:f.read(4<<20),b''):
            h.update(b)
    return h.hexdigest()

def read_jsonl(path):
    out={}
    with open(path,encoding='utf-8') as f:
        for line in f:
            if not line.strip():
                continue
            r=json.loads(line)
            out[str(r['clip_id'])]=r
    return out

def normalize_test_cache(path):
    raw=pickle.load(open(path,'rb'))
    out={}

    if isinstance(raw,dict):
        iterator=raw.items()
    else:
        iterator=((str(r['clip_id']),r) for r in raw)

    for cid,r in iterator:
        cid=str(cid)
        p=np.asarray(r.get('frame',r.get('prob',[])),np.float32).reshape(-1)
        on=np.asarray(r.get('onset',r.get('onset_dist',[])),np.float32).reshape(-1)
        off=np.asarray(r.get('offset',r.get('offset_dist',[])),np.float32).reshape(-1)
        T=min(len(p),len(on),len(off))
        dur=float(r.get('dur',r.get('duration',T*FRAME)))
        out[cid]=dict(frame=p[:T],onset=on[:T],offset=off[:T],dur=dur)

    return out

assert TEST_CACHE.is_file(),(
    f'Missing {TEST_CACHE}. Run the stable WavLM+BEATs test-posterior inference first.'
)

candidate_jsons=[

    WORK/'predictions_best.jsonl',
]
#candidate_jsons += [Path(p) for p in glob.glob(str(WORK/'**'/'*.jsonl'),recursive=True)]

BEST_JSON=None
seen=set()

for p in candidate_jsons:
    if str(p) in seen or not p.is_file():
        continue
    seen.add(str(p))
    try:
        if sha256_file(p)==EXPECTED_BEST_SHA256:
            BEST_JSON=p
            break
    except Exception:
        pass

assert BEST_JSON is not None,(
    'Exact 1.4979 predictions JSONL not found. '
    'Upload/copy it to /content/predictions.jsonl. '
    'Expected SHA256='+EXPECTED_BEST_SHA256
)

GATE_JSON=WORK/'submission_stage7_wavlm_indicw2v_fusion_diag.jsonl'
if not GATE_JSON.is_file():
    GATE_JSON=WORK/'submission_stage7_wavlm_indicw2v_fusion_diag(1).jsonl'
assert GATE_JSON.is_file(),GATE_JSON

BASE_TEST=read_jsonl(BEST_JSON)
ROUTER=read_jsonl(GATE_JSON)
TEST_POST=normalize_test_cache(TEST_CACHE)

assert len(BASE_TEST)==EXPECTED_TEST_CLIPS,(len(BASE_TEST),EXPECTED_TEST_CLIPS)
assert set(BASE_TEST)==set(ROUTER)

NAT_TEST=sorted(c for c in BASE_TEST if float(ROUTER[c]['p_syn'])<GATE_THR)
SYN_TEST=sorted(set(BASE_TEST)-set(NAT_TEST))

missing=[c for c in NAT_TEST if c not in TEST_POST]
assert not missing,f'Missing {len(missing)} Natural test clips from posterior cache; e.g. {missing[:5]}'

print('Exact best JSON:',BEST_JSON)
print('SHA256:',sha256_file(BEST_JSON))
print('Natural route:',len(NAT_TEST))
print('Synthetic frozen:',len(SYN_TEST))
print('Test posterior rows:',len(TEST_POST))


Exact best JSON: /content/drive/MyDrive/Datathon/work/predictions_best.jsonl
SHA256: bee50cc35a306ca46482e092bd842328ae8b39040e004e6a9baee0a2a47a0cfa
Natural route: 2957
Synthetic frozen: 2560
Test posterior rows: 5517


In [ ]:
# ============================================================
# 15 — Generate + score hidden Natural rescue candidates
# ============================================================

def json_events(row):
    out=[]
    for e in row.get('events',[]):
        try:
            a=float(e['onset']); b=float(e['offset'])
            if b>a:
                out.append((a,b))
        except Exception:
            pass
    return sorted(out)

TEST_BASE={
    cid:json_events(BASE_TEST[cid])
    for cid in NAT_TEST
}

TEST_CAND={}
for n,cid in enumerate(NAT_TEST,1):
    TEST_CAND[cid]=generate_candidates(TEST_POST,cid,TEST_BASE[cid])
    if n%500==0:
        print('generated',n,'/',len(NAT_TEST))

TEST_SCORED={}

for cid in NAT_TEST:
    base=TEST_BASE[cid]
    rows=[]
    idxs=[]

    for j,z in enumerate(TEST_CAND[cid]):
        src=set(z['source'].split('|'))
        if 'baseline' in src:
            continue

        ev=(z['onset'],z['offset'])
        if any(near_same(ev,b,.03) for b in base):
            continue

        rr=feature_row(TEST_POST,cid,z,base,len(TEST_CAND[cid]))
        rr['cand_idx']=j
        rows.append(rr)
        idxs.append(j)

    if not rows:
        TEST_SCORED[cid]=[]
        continue

    df=pd.DataFrame(rows)

    # Guarantee the same training feature schema.
    for c in FEATURE_COLS:
        if c not in df:
            df[c]=0

    pp=predict_rescue_proba(df)

    arr=[]
    for rr,p,j in zip(rows,pp,idxs):
        z=TEST_CAND[cid][j]
        src=set(z['source'].split('|'))
        arr.append(dict(
            event=(float(z['onset']),float(z['offset'])),
            prob=float(p),
            source=z['source'],
            is_pair=any(s.startswith('pair_') or s=='global_pair' for s in src),
            is_global_pair='global_pair' in src,
        ))

    TEST_SCORED[cid]=arr

print('Natural hidden candidates:',
      sum(len(TEST_CAND[c]) for c in NAT_TEST))
print('Natural hidden scored non-baseline candidates:',
      sum(len(TEST_SCORED[c]) for c in NAT_TEST))


generated 500 / 2957
generated 1000 / 2957
generated 1500 / 2957
generated 2000 / 2957
generated 2500 / 2957


In [ ]:
# ============================================================
# 16 — Apply TUNE-frozen rescue-only policy
# ============================================================

FINAL_NAT={}
ADD_AUDIT=[]

for cid in NAT_TEST:
    base=TEST_BASE[cid]
    final=rescue_decode(base,TEST_SCORED[cid],FROZEN_POLICY)
    FINAL_NAT[cid]=final

    base_set={(round(a,4),round(b,4)) for a,b in base}
    added=[
        (a,b) for a,b in final
        if (round(a,4),round(b,4)) not in base_set
    ]

    ADD_AUDIT.append(dict(
        clip_id=cid,
        p_syn=float(ROUTER[cid]['p_syn']),
        baseline_events=len(base),
        final_events=len(final),
        additions=len(added),
        added_events=json.dumps([
            {'onset':round(a,4),'offset':round(b,4)}
            for a,b in added
        ])
    ))

AUDIT=pd.DataFrame(ADD_AUDIT)

print('Natural route clips:',len(AUDIT))
print('Natural total baseline events:',AUDIT.baseline_events.sum())
print('Natural total additions:',AUDIT.additions.sum())
print('clips with additions:',int((AUDIT.additions>0).sum()))
print('mean additions / Natural clip:',AUDIT.additions.mean())

display(AUDIT.additions.value_counts().sort_index().rename('clips').to_frame())
AUDIT.to_csv(ROOT/'hidden_natural_addition_audit.csv',index=False)


In [ ]:
# ============================================================
# 17 — DEFINITE FINAL SUBMISSION
# Synthetic frozen exactly; Natural = exact best + rescue additions
# ============================================================

FINAL={
    cid:json.loads(json.dumps(row))
    for cid,row in BASE_TEST.items()
}

for cid in NAT_TEST:
    FINAL[cid]['events']=[
        {'onset':round(float(a),4),'offset':round(float(b),4)}
        for a,b in FINAL_NAT[cid]
    ]

# Strong invariants.
assert set(FINAL)==set(BASE_TEST)
assert len(FINAL)==EXPECTED_TEST_CLIPS

syn_changed=[
    cid for cid in SYN_TEST
    if FINAL[cid].get('events',[]) != BASE_TEST[cid].get('events',[])
]
assert not syn_changed,f'Synthetic branch changed for {len(syn_changed)} clips'

# Natural is ADD-only: every original Natural interval must still be present exactly.
natural_lost=[]
for cid in NAT_TEST:
    old={
        (round(float(e['onset']),4),round(float(e['offset']),4))
        for e in BASE_TEST[cid].get('events',[])
    }
    new={
        (round(float(e['onset']),4),round(float(e['offset']),4))
        for e in FINAL[cid].get('events',[])
    }
    if not old.issubset(new):
        natural_lost.append(cid)
assert not natural_lost,f'ADD-only invariant violated for {len(natural_lost)} Natural clips'

OUT_JSON=ROOT/'predictions_Track1_V34_1_RescueOnly_FIXED_NO_LEAK_FINAL.jsonl'
OUT_ZIP=ROOT/'submission_Track1_V34_1_RescueOnly_FIXED_NO_LEAK_FINAL.zip'
MANIFEST=ROOT/'submission_Track1_V34_1_RescueOnly_FIXED_NO_LEAK_FINAL_manifest.json'

with open(OUT_JSON,'w',encoding='utf-8') as f:
    for cid in sorted(FINAL):
        f.write(json.dumps(FINAL[cid],ensure_ascii=False)+'\n')

with zipfile.ZipFile(OUT_ZIP,'w',zipfile.ZIP_DEFLATED) as z:
    z.write(OUT_JSON,arcname='predictions.jsonl')

manifest=dict(
    version='Track1_V34_1_RescueOnly_FIXED_NO_LEAK',
    exact_best_source=str(BEST_JSON),
    exact_best_sha256=sha256_file(BEST_JSON),
    total_clips=len(FINAL),
    natural_route_clips=len(NAT_TEST),
    synthetic_frozen_clips=len(SYN_TEST),
    synthetic_changed=0,
    natural_original_events_preserved=True,
    natural_total_additions=int(AUDIT.additions.sum()),
    natural_clips_with_additions=int((AUDIT.additions>0).sum()),
    gate_threshold=GATE_THR,
    frozen_policy=FROZEN_POLICY,
    validation=EVAL.to_dict(orient='records'),
    output_json=str(OUT_JSON),
    output_zip=str(OUT_ZIP),
    output_json_sha256=sha256_file(OUT_JSON),
)
json.dump(manifest,open(MANIFEST,'w'),indent=2)

print('='*100)
print('DEFINITE V34.1 SUBMISSION CREATED')
print('='*100)
print('JSON:',OUT_JSON)
print('ZIP :',OUT_ZIP)
print('JSON SHA256:',sha256_file(OUT_JSON))
print('Synthetic changed:',len(syn_changed))
print('Natural lost original intervals:',len(natural_lost))
print('Natural additions:',int(AUDIT.additions.sum()))
print('Frozen policy:',FROZEN_POLICY)


## What to send back after running

The most useful outputs are:

```text
ADD-only oracle table
TRAIN/TUNE/SELECT/HOLD classifier AUC + PR-AUC
FROZEN TUNE-ONLY POLICY
TUNE/SELECT/HOLD validation table
Natural total additions in hidden test
```

The ZIP printed by the final cell is the submission artifact.
